In [0]:
pip install langchain langchain-community langchain-huggingface chromadb sentence-transformers

In [0]:
# pip install langchain-community langchain-text-splitters pypdf

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Load PDF
pdf_path = "/Workspace/Users/surbhiwahie@gmail.com/Rag-System---Ask-My-Doc/Data/raw/Diabetes_WHO.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

print(f"Loaded {len(documents)} pages")

In [0]:
# 2. Chunking
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks")

In [0]:
# 3. Add metadata
for i, doc in enumerate(chunks):
    doc.metadata["source"] = "Diabetes_WHO.pdf"
    doc.metadata["chunk_id"] = i

# 4. Embeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# 5. Store in Chroma
db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="/tmp/chroma_db"
)
db.persist()

print("✅ Vector DB created!")

In [0]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# 2. Load DB
db = Chroma(
    persist_directory="/tmp/chroma_db",
    embedding_function=embedding_model
)

# 3. Retriever
retriever = db.as_retriever(search_kwargs={"k": 5})

# 4. Query
query = "What are the Symptoms of Diabetes?"
results = retriever.invoke(query)

# 5. Print
for r in results:
    print("----")
    print(r.page_content[:300])
    print(r.metadata)